# Model Training - CNN 1D Architecture

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# If it fails to determine best cudnn convolution algorithm
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [ ]:
get_gpu_info()

## 2. Run Parameters 

In [ ]:
EPOCHS = 100
BATCH_SIZE = 64

DATA_SEED = 99
TRAIN_SEED = 333

# Set Python, NumPy, Keras and TensorFlow seeds
set_random_seed(TRAIN_SEED)

# Reproducibility settings for TensorFlow:
# Note: must have same inputs and hardware
# Warning: this affects overall performance
# tf.config.experimental.enable_op_determinism()

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
JIT_COMPILE = False

In [ ]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [ ]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/train_1")

## 3. Data Loading and Preprocessing

In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

In [ ]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 4. Model Definition

In [ ]:
def build_model(show_summary: bool = True) -> Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    x1 = layers.Conv1D(
        filters=256,
        kernel_size=5,
        padding="same",
        data_format="channels_last",
        activation=None,
        kernel_initializer=initializer,
        name="conv1d_1",
    )(combined)
    x1 = layers.BatchNormalization(name="conv1d_bn_1")(x1)
    x1 = layers.Activation("sigmoid", name="conv1d_act_1")(x1)
    x1 = layers.MaxPooling1D(pool_size=2, name="max_pool_1")(x1)

    x2 = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        data_format="channels_last",
        activation=None,
        kernel_initializer=initializer,
        name="conv1d_2",
    )(x1)
    x2 = layers.BatchNormalization(name="conv1d_bn_2")(x2)
    x2 = layers.Activation("relu", name="conv1d_act_2")(x2)
    x2 = layers.MaxPooling1D(pool_size=2, name="max_pool_2")(x2)

    # Align temporal length, keep channels, then concat on the last axis
    _skip12 = resize_for_skip_1d(x1, x2.shape[1], name="skip_cnn1_to_cnn2_resize")
    x2 = layers.Concatenate(axis=-1, name="skip_from_cnn1_to_cnn2")([_skip12, x2])

    x3 = layers.Conv1D(
        filters=128,
        kernel_size=7,
        padding="same",
        data_format="channels_last",
        activation=None,
        kernel_initializer=initializer,
        name="conv1d_3",
    )(x2)
    x3 = layers.BatchNormalization(name="conv1d_bn_3")(x3)
    x3 = layers.Activation("silu", name="conv1d_act_3")(x3)
    x3 = layers.MaxPooling1D(pool_size=3, name="max_pool_3")(x3)

    x4 = layers.Conv1D(
        filters=256,
        kernel_size=7,
        padding="same",
        data_format="channels_last",
        activation=None,
        kernel_initializer=initializer,
        name="conv1d_4",
    )(x3)
    x4 = layers.BatchNormalization(name="conv1d_bn_4")(x4)
    x4 = layers.Activation("gelu", name="conv1d_act_4")(x4)
    x4 = layers.MaxPooling1D(pool_size=3, name="max_pool_4")(x4)

    # ———————————————————————————————————— DNN ——————————————————————————————————— #
    x = x4
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(
        units=500,
        activation=None,
        kernel_initializer=initializer,
        name="dense_1"
    )(x)
    x = layers.Activation("sigmoid", name="dense_act_1")(x)
    x = layers.Dropout(rate=0.2)(x)

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    steps_per_epoch = max(1, len(y_train) // BATCH_SIZE)
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = max(10, int(0.05 * total_steps))
    decay_steps = max(1, total_steps - warmup_steps)

    lr_schedule = optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0,
        decay_steps=decay_steps,
        alpha=0.0,
        warmup_target=7e-5,
        warmup_steps=warmup_steps,
    )

    optimizer = optimizers.Adam(learning_rate=lr_schedule)
    
    # optimizer = optimizers.Adam(learning_rate=7e-5)

    model.compile(
        optimizer=optimizer,
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=JIT_COMPILE,
    )

    return model

## Main

In [ ]:
try:
    # ——————————————————————————————————— Setup —————————————————————————————————— #
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
        tensorboard_dir,
    ) = init_study_dirs(RUN_DIR, study_name="model_training")

    # ——————————————————————————————— Prepare Data ——————————————————————————————— #
    coord_scaler = StandardScaler()

    coord_scaler.fit(x_coord_train)
    x_coord_train = coord_scaler.transform(x_coord_train)
    x_coord_val = coord_scaler.transform(x_coord_val)
    x_coord_test = coord_scaler.transform(x_coord_test)
    s009_coord_input = coord_scaler.transform(s009_coord_input)

    # —————————————————————————————— Train the Model ————————————————————————————— #

    model = build_model(show_summary=True)
    save_model_plot(model, output_path=os.path.join(fig_dir, "model_plot.png"))

    history = model.fit(
        x=[x_lidar_train, x_coord_train],
        y=y_train,
        validation_data=([x_lidar_val, x_coord_val], y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks_model(
            backup_dir=os.path.join(backup_dir, "training"),
            checkpoint_dir=os.path.join(backup_dir, "checkpoints"),
            #! Can cause high memory usage
            # tensorboard_logs=tensorboard_dir,
            early_stopping_patience=None,
            reduce_lr_patience=None,
        ),
        verbose=2,
    )

    model.save(os.path.join(model_dir, "model.keras"))

    # ——————————————————————————————— Test on S009 ——————————————————————————————— #
    test_loss, test_acc = model.evaluate(
        [x_lidar_test, x_coord_test], y_test, batch_size=BATCH_SIZE, verbose=0
    )
    # Now evaluate on the full s009 dataset for comparison purposes
    test_loss_full, test_acc_full = model.evaluate(
        [s009_lidar_input, s009_coord_input], s009_y, batch_size=BATCH_SIZE, verbose=0
    )

    # ——————————————————————————————— Save history ——————————————————————————————— #
    history_path = os.path.join(history_dir, "history.csv")
    history_data = {
        "epoch": list(range(1, len(history.history["loss"]) + 1)),
        "train_loss": history.history["loss"],
        "val_loss": history.history["val_loss"],
        "test_loss_s009": test_loss,
        "test_acc_s009": test_acc,
        "test_loss_s009_full": test_loss_full,
        "test_acc_s009_full": test_acc_full,
    }

    history_df = pd.DataFrame(history_data)
    history_df.to_csv(history_path, index=False)

    # ———————————————————————————————— Model Stats ——————————————————————————————— #
    write_model_stats_to_file(
        model=model,
        file_path=os.path.join(args_dir, "model_stats.txt"),
        batch_size=BATCH_SIZE,
        bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
        device=0,
        n_trials=1000,
        verbose=True,
        extra_attrs={
            "final_loss": history.history["loss"][-1],
            "final_val_loss": history.history["val_loss"][-1],
            "test_loss_s009": test_loss,
            "test_acc_s009": test_acc,
            "test_loss_s009_full": test_loss_full,
            "test_acc_s009_full": test_acc_full,
        },
    )
    # ———————————————————————————————————————————————————————————————————————————— #
except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training:\n{e}\n{traceback.format_exc()}\n\n")
finally:
    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)